<a href="https://colab.research.google.com/github/geek4s/tokenizer-using-hugging-face/blob/main/Tokenizer_Comparison.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [35]:
with open("pride_prejudice.txt", "r", encoding="utf-8") as f:
    text = f.read()

print(text[:500])

The Project Gutenberg eBook of Pride and Prejudice
    
This eBook is for the use of anyone anywhere in the United States and
most other parts of the world at no cost and with almost no restrictions
whatsoever. You may copy it, give it away or re-use it under the terms
of the Project Gutenberg License included with this eBook or online
at www.gutenberg.org. If you are not located in the United States,
you will have to check the laws of the country where you are located
before using this eBook.




In [38]:
start = text.find("*** START OF THE PROJECT GUTENBERG EBOOK")
if start != -1:
    text = text[start:]
    text = text.split("\n", 1)[1]

end = text.find("*** END OF THE PROJECT GUTENBERG EBOOK")
if end != -1:
    text = text[:end]

In [39]:
import re

text = text.lower()
text = re.sub(r"[^a-z\s]", "", text)
text = re.sub(r"\s+", " ", text).strip()

In [40]:
words = text.split()

print("Total words:", len(words))

Total words: 127160


In [41]:
corpus = words[:5000]

print(len(corpus))

5000


In [42]:
corpus_text = " ".join(corpus)

with open("corpus5000.txt", "w") as f:
    f.write(corpus_text)

In [43]:
from collections import Counter

word_freq = Counter(corpus)

print(word_freq.most_common(10))

[('the', 305), ('of', 216), ('and', 173), ('to', 147), ('in', 115), ('a', 85), ('is', 72), ('not', 71), ('that', 57), ('it', 56)]


In [44]:
# Install the required libraries
!pip install -q tokenizers sentencepiece

In [45]:
# Import the libraries
from tokenizers import Tokenizer
from tokenizers.models import BPE, WordPiece
from tokenizers.trainers import BpeTrainer, WordPieceTrainer
from tokenizers.pre_tokenizers import Whitespace

import sentencepiece as spm

In [48]:
# Train the BPE Tokenizer
# Create BPE tokenizer
bpe_tokenizer = Tokenizer(BPE(unk_token="[UNK]"))

# Split text on whitespace
bpe_tokenizer.pre_tokenizer = Whitespace()

# Trainer
bpe_trainer = BpeTrainer(
    vocab_size=2000,
    special_tokens=["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"]
)

# Train on the same corpus
bpe_tokenizer.train(["corpus5000.txt"], trainer=bpe_trainer)

print("BPE training completed!")

BPE training completed!


In [54]:
import re

sentence = "Elizabeth Bennet loves reading books."

sentence = sentence.lower()
sentence = re.sub(r"[^a-z\s]", "", sentence)

encoding = bpe_tokenizer.encode(sentence)

In [56]:
vocab = bpe_tokenizer.get_vocab()

print("elizabeth" in vocab)
print("bennet" in vocab)
print("books" in vocab)
print("reading" in vocab)

True
True
False
False


In [59]:
# Test the BPE Tokenizer
sentence = "the characteristics of miss austen's humour are so subtle and delicate"

encoding = bpe_tokenizer.encode(sentence)

print("Original Sentence:")
print(sentence)

print("\nTokens:")
print(encoding.tokens)

print("\nToken IDs:")
print(encoding.ids)

Original Sentence:
the characteristics of miss austen's humour are so subtle and delicate

Tokens:
['the', 'characteristics', 'of', 'miss', 'austen', '[UNK]', 's', 'humour', 'are', 'so', 'subtle', 'and', 'delicate']

Token IDs:
[34, 1245, 39, 124, 135, 1, 23, 667, 120, 193, 1308, 44, 1248]


In [60]:
bpe_tokenizer.save("bpe_tokenizer.json")

In [61]:
# Train the WordPiece Tokenizer
wordpiece_tokenizer = Tokenizer(
    WordPiece(unk_token="[UNK]")
)

wordpiece_tokenizer.pre_tokenizer = Whitespace()

wordpiece_trainer = WordPieceTrainer(
    vocab_size=1500,
    special_tokens=["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"]
)

wordpiece_tokenizer.train(
    ["corpus5000.txt"],
    trainer=wordpiece_trainer
)

print("WordPiece training completed!")

WordPiece training completed!


In [63]:
# Test the WordPiece Tokenizer
sentence = "when miss austen was barely twenty-one; though it was revised and finished at chawton some fifteen years later, and was not published till 1813"

encoding = wordpiece_tokenizer.encode(sentence)

print("Original Sentence:")
print(sentence)

print("\nTokens:")
print(encoding.tokens)

print("\nToken IDs:")
print(encoding.ids)

Original Sentence:
when miss austen was barely twenty-one; though it was revised and finished at chawton some fifteen years later, and was not published till 1813

Tokens:
['when', 'miss', 'austen', 'was', 'barely', 'tw', '##ent', '##y', '[UNK]', 'one', '[UNK]', 'though', 'it', 'was', 'revis', '##ed', 'and', 'fin', '##ished', 'at', 'ch', '##aw', '##ton', 'some', 'fi', '##ft', '##een', 'years', 'later', '[UNK]', 'and', 'was', 'not', 'publish', '##ed', 't', '##ill', '[UNK]']

Token IDs:
[232, 152, 164, 170, 1361, 371, 106, 47, 1, 186, 1, 299, 92, 170, 1222, 76, 68, 550, 1267, 134, 99, 568, 683, 424, 287, 1128, 295, 625, 1366, 1, 68, 170, 95, 1421, 76, 24, 227, 1]


In [64]:
wordpiece_tokenizer.save("wordpiece_tokenizer.json")

In [28]:
# Train the Unigram Tokenizer (SentencePiece)
import sentencepiece as spm

# Read the content of the existing corpus5000.txt
with open("corpus5000.txt", "r", encoding="utf-8") as f:
    single_line_text = f.read()

# Split the text into words and join them with newline characters
# This creates a format where each "sentence" (in this case, each word) is on its own line,
# which SentencePieceTrainer expects.
words = single_line_text.split()
formatted_text = "\n".join(words)

# Write the formatted text to a new temporary file
formatted_input_filename = "corpus5000_formatted.txt"
with open(formatted_input_filename, "w", encoding="utf-8") as f:
    f.write(formatted_text)

# Now, train SentencePiece using the newly formatted file
spm.SentencePieceTrainer.train(
    input=formatted_input_filename,
    model_prefix="unigram",
    vocab_size=1500,
    model_type="unigram"
)

print("Unigram training completed!")

Unigram training completed!


In [65]:
# Load the Unigram Model
sp = spm.SentencePieceProcessor()
sp.load("unigram.model")

True

In [66]:
# Test the Unigram Tokenizer
sentence = "elizabeth bennet loves reading."

tokens = sp.encode(sentence, out_type=str)
ids = sp.encode(sentence, out_type=int)

print("Original Sentence:")
print(sentence)

print("\nTokens:")
print(tokens)

print("\nToken IDs:")
print(ids)

Original Sentence:
elizabeth bennet loves reading.

Tokens:
['▁', 'eli', 'z', 'abeth', '▁bennet', '▁love', 's', '▁read', 'ing', '.']

Token IDs:
[3, 160, 0, 184, 143, 170, 4, 380, 16, 0]


In [67]:
# Compare All Three Tokenizers
sentence = "elizabeth bennet loves reading."

print("="*60)
print("Sentence:")
print(sentence)

print("\nBPE")
print(bpe_tokenizer.encode(sentence).tokens)

print("\nWordPiece")
print(wordpiece_tokenizer.encode(sentence).tokens)

print("\nUnigram")
print(sp.encode(sentence, out_type=str))

Sentence:
elizabeth bennet loves reading.

BPE
['elizabeth', 'bennet', 'loves', 'read', 'ing', '[UNK]']

WordPiece
['elizabeth', 'bennet', 'love', '##s', 'read', '##ing', '[UNK]']

Unigram
['▁', 'eli', 'z', 'abeth', '▁bennet', '▁love', 's', '▁read', 'ing', '.']


In [68]:
# Compare Vocabulary Sizes
print("Vocabulary Sizes")
print("----------------------------")

print("BPE:", bpe_tokenizer.get_vocab_size())
print("WordPiece:", wordpiece_tokenizer.get_vocab_size())
print("Unigram:", sp.get_piece_size())

Vocabulary Sizes
----------------------------
BPE: 2000
WordPiece: 1500
Unigram: 500


In [69]:
# Compare Number of Tokens Produced
test_sentences = [
    "Elizabeth Bennet loves reading books.",
    "It is a truth universally acknowledged.",
    "The quick brown fox jumps over the lazy dog.",
    "Artificial intelligence is transforming technology."
]

print("{:<50} {:<8} {:<12} {:<10}".format("Sentence", "BPE", "WordPiece", "Unigram"))
print("-"*90)

for s in test_sentences:

    bpe = len(bpe_tokenizer.encode(s).tokens)
    wp = len(wordpiece_tokenizer.encode(s).tokens)
    uni = len(sp.encode(s, out_type=str))

    print(f"{s[:45]:<50} {bpe:<8} {wp:<12} {uni:<10}")

Sentence                                           BPE      WordPiece    Unigram   
------------------------------------------------------------------------------------------
Elizabeth Bennet loves reading books.              13       9            18        
It is a truth universally acknowledged.            10       12           19        
The quick brown fox jumps over the lazy dog.       20       21           26        
Artificial intelligence is transforming techn      22       14           34        
